<a href="https://colab.research.google.com/github/majd-almohsen/multi-task-eye-disease-detection-vit/blob/main/multi_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# أولاً: تنظيف شامل
import os, shutil, gc, torch

# أغلاق أي توصيل سابق
try:
    from google.colab import drive
    drive.flush_and_unmount()
except:
    pass

# تنظيف الذاكرة
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# الآن وصّل Drive
print("جارٍ توصيل Google Drive...")
drive.mount('/content/drive', force_remount=True)
print("تم التوصيل بنجاح!")

Drive not mounted, so nothing to flush and unmount.
جارٍ توصيل Google Drive...
Mounted at /content/drive
تم التوصيل بنجاح!


In [ ]:
!pip install transformers==4.57.6
import os
os.kill(os.getpid(), 9)

In [ ]:
#الإصدارات المتوافقة:
#TensorFlow==2.19.0
#Keras==3.10.0
#transformers==4.57.6

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from transformers import TFViTModel
import cv2
import numpy as np
from PIL import Image
import os
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

# ======================== 1. دالة قص الصورة ========================
def crop_image(image_path, threshold=30):
    try:
        # تحميل الصورة
        img = Image.open(image_path).convert('RGB')
        width, height = img.size
        pixels = img.load()

        # إيجاد المحتوى غير الأسود
        left, right, top, bottom = width, 0, height, 0

        for y in range(height):
            for x in range(width):
                r, g, b = pixels[x, y]
                if r > threshold or g > threshold or b > threshold:
                    left = min(left, x)
                    right = max(right, x)
                    top = min(top, y)
                    bottom = max(bottom, y)

        if left < right and top < bottom:
            cropped_img = img.crop((left, top, right+1, bottom+1))
        else:
            cropped_img = img

        # تحويل إلى numpy array
        return np.array(cropped_img)

    except Exception as e:
        print(f"Error cropping image {image_path}: {e}")
        return None

# ======================== 2. دالة توحيد الألوان ========================
def unify_colors(image_array):
    try:
        # 1. تصحيح توازن الألوان
        img_float = image_array.astype(np.float32) / 255.0
        mean_r = np.mean(img_float[:,:,0])
        mean_g = np.mean(img_float[:,:,1])
        mean_b = np.mean(img_float[:,:,2])
        gray_mean = (mean_r + mean_g + mean_b) / 3.0

        r_gain = gray_mean / (mean_r + 1e-7)
        g_gain = gray_mean / (mean_g + 1e-7)
        b_gain = gray_mean / (mean_b + 1e-7)

        img_corrected = img_float.copy()
        img_corrected[:,:,0] = np.clip(img_corrected[:,:,0] * r_gain, 0, 1)
        img_corrected[:,:,1] = np.clip(img_corrected[:,:,1] * g_gain, 0, 1)
        img_corrected[:,:,2] = np.clip(img_corrected[:,:,2] * b_gain, 0, 1)
        img_corrected = (img_corrected * 255).astype(np.uint8)

        # 2. تحسين التباين في مساحة LAB
        img_rgb = cv2.cvtColor(img_corrected, cv2.COLOR_BGR2RGB)
        lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        l_enhanced = clahe.apply(l)

        enhanced_lab = cv2.merge((l_enhanced, a, b))
        enhanced_rgb = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2RGB)
        enhanced_bgr = cv2.cvtColor(enhanced_rgb, cv2.COLOR_RGB2BGR)

        return enhanced_bgr

    except Exception as e:
        print(f"Error unifying colors: {e}")
        return image_array

# ======================== 3. دالة معالجة الصورة ========================
def preprocess_image(image_array, target_size=(224, 224)):
    try:
        # تغيير الحجم
        resized = cv2.resize(image_array, target_size)

        # تحويل BGR إلى RGB إذا لزم
        if len(resized.shape) == 3 and resized.shape[2] == 3:
            resized = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)

        # تطبيع للقيم بين 0 و1
        normalized = resized.astype(np.float32) / 255.0

        # إضافة بُعد الدُفعة
        processed = np.expand_dims(normalized, axis=0)

        return processed

    except Exception as e:
        print(f"Error preprocessing image: {e}")
        return None

# ======================== 4. بناء النموذج متعدد المهام ========================
# طبقة ViT المخصصة
class ViTLayer(layers.Layer):
    def __init__(self, vit_model, **kwargs):  # التغيير هنا: استخدام vit_model بدلًا من vit_config
        super().__init__(**kwargs)
        self.vit_model = vit_model  # تخزين النموذج مباشرةً

    def call(self, inputs):
        inputs = tf.transpose(inputs, perm=[0, 3, 1, 2])
        return self.vit_model(inputs).last_hidden_state[:, 0, :]

    def get_config(self):
        config = super().get_config()
        # حفظ تكوين النموذج (ليس الأوزان)
        config["vit_config"] = self.vit_model.config.to_dict()
        return config

    @classmethod
    def from_config(cls, config):
        # إعادة بناء النموذج من التكوين (مع تجميد الأوزان)
        vit_config = TFViTModel.config_class.from_dict(config["vit_config"])
        vit_model = TFViTModel.from_config(vit_config)
        vit_model.trainable = False
        return cls(vit_model=vit_model)

def build_multi_task_model():

    MODEL_NAME = "google/vit-large-patch16-224"
    # تحميل ViT
    base_model = TFViTModel.from_pretrained(MODEL_NAME)

    # بناء النموذج
    input_layer = layers.Input(shape=(224, 224, 3), name="input_image")

    # طبقة ViT المخصصة
    vit_layer = ViTLayer(vit_model=base_model)
    vit_output = vit_layer(input_layer)
    shared_features = vit_output

    # 4. الرأس الأول: تصنيف نوع المرض (3 فئات)
    head1 = layers.BatchNormalization(name="head1_bn1")(shared_features)
    head1 = layers.Dense(512, activation='relu', name="head1_dense1")(head1)
    head1 = layers.Dropout(0.5, name="head1_dropout1")(head1)
    head1 = layers.Dense(256, activation='relu', name="head1_dense2")(head1)
    head1 = layers.Dropout(0.4, name="head1_dropout2")(head1)
    head1 = layers.Dense(64, activation='relu', name="head1_dense3")(head1)
    head1 = layers.Dropout(0.3, name="head1_dropout3")(head1)
    head1_output = layers.Dense(3, activation='softmax', name="head1_output")(head1)

    # 5. الرأس الثاني: شدة السكري (2 فئات)
    head2 = layers.BatchNormalization(name="head2_bn1")(shared_features)
    head2 = layers.Dense(256, activation='relu', name="head2_dense1")(head2)
    head2 = layers.Dropout(0.5, name="head2_dropout1")(head2)
    head2 = layers.Dense(128, activation='relu', name="head2_dense2")(head2)
    head2 = layers.Dropout(0.4, name="head2_dropout2")(head2)
    head2 = layers.Dense(64, activation='relu', name="head2_dense3")(head2)
    head2 = layers.Dropout(0.3, name="head2_dropout3")(head2)
    head2_output = layers.Dense(1, activation='sigmoid', name="head2_output")(head2)

    # بناء النموذج النهائي
    model = Model(
        inputs=input_layer,
        outputs=[head1_output, head2_output],
        name="ViT_MultiTask_Full"
    )

    return model

# تحميل النموذج مع الأوزان
def load_model_with_weights(model_weights_path):
    model = build_multi_task_model()
    model.load_weights(model_weights_path)

    # تجميد جميع الطبقات
    for layer in model.layers:
        layer.trainable = False

    print("✅ Model loaded successfully")
    return model

def predict_image(model, image_path):
    try:
        # 1. قص الصورة
        cropped = crop_image(image_path)
        if cropped is None:
            return {"error": "Failed to crop image"}

        # 2. توحيد الألوان
        unified = unify_colors(cropped)

        # 3. معالجة الصورة
        processed = preprocess_image(unified)
        if processed is None:
            return {"error": "Failed to preprocess image"}

        # 4. التنبؤ
        predictions = model.predict(processed, verbose=0)

        # تفسير النتائج
        disease_probs = predictions[0][0]  # الرأس الأول
        severity_prob = predictions[1][0][0]  # الرأس الثاني
        # تحديد فئة المرض
        disease_classes = ['Normal', 'Diabetes', 'Other Diseases']
        disease_idx = np.argmax(disease_probs)
        disease_label = disease_classes[disease_idx]
        disease_confidence = disease_probs[disease_idx]

        # إنشاء النتيجة الأساسية
        result = {
            "disease_type": {
                "label": disease_label,
                "confidence": float(disease_confidence),
                "all_probabilities": {
                    'Normal': float(disease_probs[0]),
                    'Diabetes': float(disease_probs[1]),
                    'Other Diseases': float(disease_probs[2])
                }
            },
            "processed_image": processed[0]  # للعرض
        }

        # ✅ فقط إذا كان التشخيص Diabetes نعرض نتيجة الرأس الثاني
        if disease_label == 'Diabetes':
            severity_label = "Severe" if severity_prob > 0.5 else "Mild/Moderate"
            severity_confidence = severity_prob if severity_prob > 0.5 else 1 - severity_prob

            result["diabetes_severity"] = {
                "label": severity_label,
                "confidence": float(severity_confidence),
                "probability": float(severity_prob)
            }
        else:
            # ✅ إذا لم يكن Diabetes
            result["diabetes_severity"] = {
                "label": "Not found",
                "confidence": 0.0,
                "probability": 0.0,
                "note": "Severity assessment is only available for Diabetes cases"
            }

        return result

    except Exception as e:
        return {"error": f"Prediction failed: {str(e)}"}


# واجهة برمجية متكاملة لتشخيص أمراض العين
class OcularDiseaseDiagnosisUI:
    def __init__(self, model_weights_path):

        self.model = load_model_with_weights(model_weights_path)
        self._setup_ui()

    def _setup_ui(self):
        # أزرار التحميل والتحليل
        self.uploader = widgets.FileUpload(
            description='Upload Eye Image',
            accept='image/*',
            multiple=False
        )
        self.analyze_button = widgets.Button(
            description='Analyze Image',
            button_style='success',
            icon='stethoscope'
        )
        self.analyze_button.on_click(self._analyze_image)

        # عناصر عرض الصور
        self.original_image = widgets.Image(
            format='jpeg',
            width=250,
            height=250,
            description='Original:'
        )

        self.processed_image = widgets.Image(
            format='jpeg',
            width=250,
            height=250,
            description='Processed:'
        )

        # عناصر عرض النتائج
        self.disease_result = widgets.HTML(
            value="<h3>Disease Type:</h3><p>Waiting for analysis...</p>"
        )

        self.severity_result = widgets.HTML(
            value="<h3>Diabetes Severity:</h3><p>Waiting for analysis...</p>"
        )

        # شريط التقدم
        self.progress = widgets.FloatProgress(
            value=0,
            min=0,
            max=100,
            description='Progress:',
            bar_style='info'
        )

        # إنشاء التخطيط
        self.ui = widgets.VBox([
            widgets.HTML("<h1 style='text-align:center; color:#2c3e50;'>ߑo؏ Ocular Disease Diagnosis System</h1>"),
            widgets.HTML("<p style='text-align:center; color:#555;'>Upload a fundus image for multi-task diagnosis</p>"),

            widgets.HBox([
                widgets.VBox([
                    widgets.HTML("<h4>Original Image</h4>"),
                    self.original_image
                ]),
                widgets.VBox([
                    widgets.HTML("<h4>Processed Image</h4>"),
                    self.processed_image
                ])
            ], layout=widgets.Layout(justify_content='space-around')),

            self.uploader,
            self.analyze_button,
            self.progress,
            widgets.HBox([
                widgets.VBox([
                    self.disease_result
                ], layout=widgets.Layout(width='50%', padding='10px')),
                widgets.VBox([
                    self.severity_result
                ], layout=widgets.Layout(width='50%', padding='10px'))
            ]),

            widgets.HTML("<hr><p style='color:#7f8c8d; text-align:center;'>Hybrid ViT Multi-Task Diagnosis System</p>")
        ], layout=widgets.Layout(
            border='2px solid #3498db',
            border_radius='10px',
            padding='20px',
            background_color='#ecf0f1'
        ))

    def _update_progress(self, value, message=""):
        """تحديث شريط التقدم"""
        self.progress.value = value
        if message:
            self.progress.description = f'Progress: {message}'

    def _analyze_image(self, change):
        clear_output(wait=True)
        display(self.ui)

        try:
          if not self.uploader.value:
            self.disease_result.value = "<h3>Disease Type:</h3><p style='color:red;'>Please upload an image first</p>"
            return

          # تحديث التقدم
          self._update_progress(10, "Loading image...")

          # تحميل الصورة
          uploaded_files = list(self.uploader.value.values())
          uploaded_file = uploaded_files[0]
          image_bytes = uploaded_file['content']

          # عرض الصورة الأصلية
          self.original_image.value = image_bytes

          # حفظ مؤقت
          temp_path = "/tmp/temp_eye_image.jpg"
          with open(temp_path, 'wb') as f:
            f.write(image_bytes)

          # تحديث التقدم
          self._update_progress(30, "Cropping image...")

          # التنبؤ
          results = predict_image(self.model, temp_path)

          if "error" in results:
            error_msg = f"<h3>Error:</h3><p style='color:red;'>{results['error']}</p>"
            self.disease_result.value = error_msg
            self._update_progress(0)
            return

          # تحديث التقدم
          self._update_progress(70, "Analyzing results...")

          # عرض الصورة المعالجة
          processed_img = (results['processed_image'] * 255).astype(np.uint8)
          _, encoded_img = cv2.imencode('.jpg', processed_img)
          self.processed_image.value = encoded_img.tobytes()

          # عرض نتائج المرض
          disease = results['disease_type']
          disease_html = f"""
          <h3 style='color:#2c3e50;'>Disease Type Diagnosis</h3>
          <div style='background-color:{'#d4edda' if disease['label']=='Normal' else '#f8d7da' if disease['label']=='Diabetes' else '#cce5ff'}; padding:10px; border-radius:5px;'>
            <h4 style='color:{'#155724' if disease['label']=='Normal' else '#721c24' if disease['label']=='Diabetes' else '#004085'};'>
                Diagnosis: {disease['label']} ({disease['confidence']:.1%})
            </h4>
            <p><strong>Probabilities:</strong></p>
            <ul>
                <li>Normal: {disease['all_probabilities']['Normal']:.1%}</li>
                <li>Diabetes: {disease['all_probabilities']['Diabetes']:.1%}</li>
                <li>Other Diseases: {disease['all_probabilities']['Other Diseases']:.1%}</li>
            </ul>
          </div>
          """
          self.disease_result.value = disease_html

          # ✅ التعديل هنا: عرض نتائج الشدة بناءً على التسمية الجديدة
          severity = results['diabetes_severity']

          if severity['label'] == 'Not found':
            severity_html = f"""
            <h3 style='color:#2c3e50;'>Diabetes Severity Assessment</h3>
            <div style='background-color:#e9ecef; padding:10px; border-radius:5px;'>
                <p style='color:#6c757d;'><strong>Not applicable</strong><br>
                {severity.get('note', 'Severity assessment is only available for Diabetes cases')}</p>
            </div>
            """
          else:
            # تحديد اللون حسب الشدة
            bg_color = '#fff3cd' if severity['label'] == 'Mild/Moderate' else '#f8d7da'
            text_color = '#856404' if severity['label'] == 'Mild/Moderate' else '#721c24'

            severity_html = f"""
            <h3 style='color:#2c3e50;'>Diabetes Severity Assessment</h3>
            <div style='background-color:{bg_color}; padding:10px; border-radius:5px;'>
                <h4 style='color:{text_color};'>
                    Severity: {severity['label']} ({severity['confidence']:.1%})
                </h4>
                <p>Probability of severe diabetes: {severity['probability']:.1%}</p>
            </div>
            """

          self.severity_result.value = severity_html

          # تحديث التقدم النهائي
          self._update_progress(100, "Analysis complete!")

          # تنظيف الملف المؤقت
          try:
             os.remove(temp_path)
          except:
             pass

        except Exception as e:
          error_html = f"""
          <h3 style='color:#721c24;'>Analysis Error</h3>
          <div style='background-color:#f8d7da; padding:10px; border-radius:5px;'>
             <p>{str(e)}</p>
          </div>
          """
          self.disease_result.value = error_html
          self._update_progress(0)

    def run(self):
        """تشغيل الواجهة"""
        display(self.ui)

# ======================== 7. الدالة الرئيسية ========================
def main():
    """
    الدالة الرئيسية لتشغيل النظام الكامل
    """
    print("="*60)
    print(" Ocular Disease Diagnosis System - Multi-Task ViT Model")
    print("="*60)
    print("\nSystem Components:")
    print("1. Image cropping function")
    print("2. Color unification function")
    print("3. Image preprocessing function")
    print("4. Multi-task ViT model with two heads")
    print("5. Interactive GUI for diagnosis")
    print("\n" + "="*60)

    # مسار أوزان النموذج - غير هذا حسب مسارك
    model_weights_path = "/content/drive/MyDrive/final_experience/ViT_MultiTask_weights.weights.h5"

    # التحقق من وجود الأوزان
    if not os.path.exists(model_weights_path):
        print(f"❌ Model weights not found at: {model_weights_path}")
        print("Please update the model_weights_path variable with the correct path.")
        return

    # تشغيل الواجهة
    print("\nߚ Launching diagnosis system...")
    app = OcularDiseaseDiagnosisUI(model_weights_path)
    app.run()

if __name__ == "__main__":
        main()
